# spi_nb_silver_tax

**Purpose.** Silver transformation of the AEAT receipts by office for the Tax
source (source 5). Finds the header row by content, maps columns by label,
drops the trailing empty rows, casts the values, and checks the office and
chapter identities the sheet carries. Keeps four net tax concepts plus a
computed `Resto Ingresos netos`, so every row is additive, for Nacional,
Madrid and Cataluña (ADR-017).

**Inputs.** Delta table `spi_bronze_tax_raw` in the default Lakehouse: the
sheet `datos_delegaciones` as received, header row included, positional
columns `col_01` … `col_76`, all string (ADR-013, 2026-10-03 addendum).

**Outputs.** Delta table `spi_silver_tax_cleaned` in the default Lakehouse,
written with `mode="overwrite"`. Columns: `region_name`, `indicator_category`,
`year`, `month`, `indicator_value`, `_ingestion_timestamp`. Values in thousand
EUR.

**Dependencies.** `pandas`, `deltalake`. Runs on the Fabric **Python** runtime
(no Spark session), per ADR-012 (2026-10-04 addendum).

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>}`. On failure, the exception propagates uncaught, per
the 2026-09-30 addendum to ADR-009.

**Last updated.** 2026-10-06.

In [ ]:
import json
import logging
import re

import pandas as pd
from deltalake import DeltaTable, write_deltalake

# ────────────────────────────────────────────────────────────────────────────
# Tables — OneLake abfss:// URL, not /lakehouse/default/ (lessons-learned.md,
# Python notebooks)
# ────────────────────────────────────────────────────────────────────────────

TABLES_URI = (
    "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
    "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables"
)
SOURCE_TABLE = f"{TABLES_URI}/spi_bronze_tax_raw"
TARGET_TABLE = f"{TABLES_URI}/spi_silver_tax_cleaned"

# ────────────────────────────────────────────────────────────────────────────
# Layout constants (ADR-017)
# ────────────────────────────────────────────────────────────────────────────

# The header row is found by content, not position: Bronze stores no row
# order (ADR-017 Decision 4, ADR-016 Consequences)
HEADER_MARKER = "Ejercicio"

YEAR_LABEL = "Ejercicio"
MONTH_LABEL = "Mes"
CONCEPT_LABEL = "Concepto"

CATALUNA_PROVINCES = ["Barcelona", "Gerona", "Lérida", "Tarragona"]

VALUE_LABELS = [
    "Total",
    "D.E. Madrid",
    "D.E. Cataluña",
    *CATALUNA_PROVINCES,
    "Delegaciones",
    "Servicios Centrales",
]
REQUIRED_LABELS = [YEAR_LABEL, MONTH_LABEL, CONCEPT_LABEL, *VALUE_LABELS]

# Header label -> spi_dim_region.region_name. Values copied from
# spi_df_gold_dimensions, not retyped (ADR-010 §4). Nacional = Total =
# Delegaciones + Servicios Centrales (ADR-017 Decision 2).
REGION_MAP = {
    "Total": "Nacional",
    "D.E. Madrid": "Madrid",
    "D.E. Cataluña": "Cataluña",
}

# Concept labels copied from the source text (ADR-017 Decision 1)
TOTAL_CONCEPT = "Total Ingresos netos"
TAX_CONCEPTS = [
    "IRPF Ingresos netos",
    "IVA Ingresos netos",
    "I.SOCIEDADES Ingresos netos",
    "II.EE. Ingresos netos",
]
CHAPTER_CONCEPTS = ["CAP.I Ingresos netos", "CAP.II Ingresos netos", "CAP.III Ingresos netos"]
RESTO_CATEGORY = "Resto Ingresos netos"  # computed in Silver: Total minus the four taxes

CONCEPTS_PER_PERIOD = 45

TOLERANCE = 0.5  # thousand EUR

OUTPUT_COLUMNS = [
    "region_name",
    "indicator_category",
    "year",
    "month",
    "indicator_value",
    "_ingestion_timestamp",
]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("silver_tax")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function storage_options and read_bronze
# ────────────────────────────────────────────────────────────────────────────

def storage_options() -> dict:
    """
    Build the OneLake storage options for deltalake.
    Returns:
        dict: Bearer token and Fabric endpoint flag.
    """
    return {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }


def read_bronze() -> pd.DataFrame:
    """
    Read spi_bronze_tax_raw into pandas.
    Returns:
        pd.DataFrame: The Bronze table, all columns string.
    """
    df = DeltaTable(SOURCE_TABLE, storage_options=storage_options()).to_pandas()
    log.info(f"Read spi_bronze_tax_raw: {len(df)} rows")
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create functions map_header and data_rows
# ────────────────────────────────────────────────────────────────────────────

def map_header(bronze: pd.DataFrame) -> tuple:
    """
    Find the header row by content and map the required labels to columns.
    Args:
        bronze (pd.DataFrame): The Bronze table.
    Returns:
        tuple: The header row's index label, and label -> positional column.
    Raises:
        ValueError: On zero or several header rows, or a required label
        missing or repeated.
    """
    header_rows = bronze.index[bronze["col_01"] == HEADER_MARKER]
    if len(header_rows) != 1:
        raise ValueError(f"Expected 1 header row (col_01 = '{HEADER_MARKER}'), found {len(header_rows)}")
    header_index = header_rows[0]

    positional = [c for c in bronze.columns if re.fullmatch(r"col_\d{2}", c)]
    header = bronze.loc[header_index, positional]
    columns = {}
    for label in REQUIRED_LABELS:
        found = header.index[header == label].tolist()
        if len(found) != 1:
            raise ValueError(f"Header label '{label}': expected 1 column, found {len(found)}")
        columns[label] = found[0]
    log.info(f"Header mapped: {columns}")
    return header_index, columns


def data_rows(bronze: pd.DataFrame, header_index, columns: dict) -> pd.DataFrame:
    """
    Drop the header and the trailing empty rows, keep the required columns
    under their labels, and cast them.
    Args:
        bronze (pd.DataFrame): The Bronze table.
        header_index: Index label of the header row.
        columns (dict): label -> positional column.
    Returns:
        pd.DataFrame: One row per (year, month, Concepto), values float64.
    Raises:
        ValueError: On a blank required cell in a data row. A non-numeric
        value raises from pd.to_numeric.
    """
    positional = [c for c in bronze.columns if re.fullmatch(r"col_\d{2}", c)]
    rows = bronze.drop(index=header_index)

    # Trailing empty rows: every cell '' or NULL (ADR-013, 2026-10-03 addendum)
    empty = (rows[positional].isna() | (rows[positional] == "")).all(axis=1)
    rows = rows[~empty]
    log.info(f"Dropped empty rows: {int(empty.sum())}")

    df = rows[list(columns.values())].copy()
    df.columns = list(columns)
    blank = df.isna() | (df == "")
    if blank.any().any():
        where = blank.stack()
        raise ValueError(f"Blank cells in data rows: {where[where].index.tolist()[:10]}")

    df["year"] = df[YEAR_LABEL].astype("int64")
    df["month"] = df[MONTH_LABEL].astype("int64")
    df[VALUE_LABELS] = df[VALUE_LABELS].apply(pd.to_numeric, errors="raise").astype("float64")
    df["_ingestion_timestamp"] = rows["_ingestion_timestamp"]
    log.info(f"Data rows: {len(df)}")
    return df.reset_index(drop=True)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create functions check_rows, check_offices and check_chapters
# ────────────────────────────────────────────────────────────────────────────

def check_rows(df: pd.DataFrame) -> None:
    """
    45 concepts per period, and (year, month, Concepto) unique.
    Args:
        df (pd.DataFrame): Data rows.
    Raises:
        ValueError: On a period with other than 45 concepts, or a duplicate.
    """
    keys = ["year", "month", CONCEPT_LABEL]
    duplicated = df.duplicated(keys, keep=False)
    if duplicated.any():
        sample = df.loc[duplicated, keys].drop_duplicates().head(10).to_dict("records")
        raise ValueError(f"Duplicate (year, month, Concepto): {sample}")

    counts = df.groupby(["year", "month"]).size()
    bad = counts[counts != CONCEPTS_PER_PERIOD]
    if not bad.empty:
        raise ValueError(f"Periods without {CONCEPTS_PER_PERIOD} concepts: {bad.head(10).to_dict()}")
    log.info(f"{len(counts)} periods x {CONCEPTS_PER_PERIOD} concepts, keys unique")


def check_offices(df: pd.DataFrame) -> None:
    """
    Guard: Total = Delegaciones + Servicios Centrales, and D.E. Cataluña =
    its four provinces, on every row.
    Args:
        df (pd.DataFrame): Data rows, values cast.
    Raises:
        ValueError: On a mismatch above TOLERANCE.
    """
    identities = {
        "Total = Delegaciones + Servicios Centrales":
            df["Total"] - df["Delegaciones"] - df["Servicios Centrales"],
        "D.E. Cataluña = " + " + ".join(CATALUNA_PROVINCES):
            df["D.E. Cataluña"] - df[CATALUNA_PROVINCES].sum(axis=1),
    }
    for name, residual in identities.items():
        bad = df[residual.abs() > TOLERANCE]
        if not bad.empty:
            raise ValueError(f"{name} fails: {bad[['year', 'month', CONCEPT_LABEL]].head(10).to_dict('records')}")
        log.info(f"{name}: holds on {len(df)} rows (max diff {residual.abs().max():.6f})")


def net_by_period(df: pd.DataFrame, concepts: list) -> pd.DataFrame:
    """
    Pivot the given concepts to columns, one row per (label, year, month,
    _ingestion_timestamp), for the three kept columns.
    Args:
        df (pd.DataFrame): Data rows, values cast.
        concepts (list): Concepto values to pivot.
    Returns:
        pd.DataFrame: One column per concept.
    Raises:
        ValueError: If a concept is missing in some period.
    """
    rows = df[df[CONCEPT_LABEL].isin(concepts)]
    long = rows.melt(
        id_vars=["year", "month", "_ingestion_timestamp", CONCEPT_LABEL],
        value_vars=list(REGION_MAP),
        var_name="label",
        value_name="value",
    )
    wide = long.pivot(
        index=["label", "year", "month", "_ingestion_timestamp"],
        columns=CONCEPT_LABEL,
        values="value",
    ).reindex(columns=concepts)
    if wide.isna().any().any():
        missing = wide.columns[wide.isna().any()].tolist()
        raise ValueError(f"Concepts missing in some periods: {missing}")
    return wide


def check_chapters(df: pd.DataFrame) -> None:
    """
    Guard: for Total, D.E. Madrid and D.E. Cataluña, Total Ingresos netos =
    CAP.I + CAP.II + CAP.III Ingresos netos, every period.
    Args:
        df (pd.DataFrame): Data rows, values cast.
    Raises:
        ValueError: On a mismatch above TOLERANCE.
    """
    wide = net_by_period(df, [TOTAL_CONCEPT, *CHAPTER_CONCEPTS])
    residual = (wide[TOTAL_CONCEPT] - wide[CHAPTER_CONCEPTS].sum(axis=1)).abs()
    bad = residual[residual > TOLERANCE]
    if not bad.empty:
        raise ValueError(f"{TOTAL_CONCEPT} != CAP.I + CAP.II + CAP.III: {bad.index[:10].tolist()}")
    log.info(f"Chapters identity holds for {len(wide)} (column, period) pairs (max diff {residual.max():.6f})")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function transform
# ────────────────────────────────────────────────────────────────────────────

def transform(bronze: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the Silver rules to the Bronze rows (ADR-017).
    Args:
        bronze (pd.DataFrame): The Bronze table.
    Returns:
        pd.DataFrame: The Silver DataFrame.
    Raises:
        ValueError: From the parsing functions and guards, or on a duplicate
        (indicator_category, region_name, year, month).
    """
    # 1. Header by content, columns by label, data rows cast
    header_index, columns = map_header(bronze)
    df = data_rows(bronze, header_index, columns)

    # 2. Row checks and guards
    check_rows(df)
    check_offices(df)
    check_chapters(df)

    # 3. Four net taxes plus Resto = Total - the four (ADR-017 Decision 1).
    #    Total Ingresos netos is not stored.
    wide = net_by_period(df, [TOTAL_CONCEPT, *TAX_CONCEPTS])
    wide[RESTO_CATEGORY] = wide[TOTAL_CONCEPT] - wide[TAX_CONCEPTS].sum(axis=1)
    wide = wide.drop(columns=TOTAL_CONCEPT)

    # 4. Long format with the region vocabulary. Negative values are kept
    #    (ADR-017 Decision 3): no null, zero or negative filter.
    out = (
        wide.stack()
        .rename("indicator_value")
        .reset_index()
        .rename(columns={CONCEPT_LABEL: "indicator_category"})
    )
    out["region_name"] = out["label"].map(REGION_MAP)

    # 5. (indicator_category, region_name, year, month) must be unique. Bronze
    #    is overwritten each run, so a duplicate is a defect: raise, do not
    #    deduplicate.
    keys = ["indicator_category", "region_name", "year", "month"]
    duplicated = out.duplicated(keys, keep=False)
    if duplicated.any():
        sample = out.loc[duplicated, keys].drop_duplicates().head(10).to_dict("records")
        raise ValueError(f"Duplicate (indicator_category, region_name, year, month): {sample}")

    out["indicator_value"] = out["indicator_value"].astype("float64")
    out["_ingestion_timestamp"] = (
        pd.to_datetime(out["_ingestion_timestamp"], utc=True).astype("datetime64[us, UTC]")
    )

    return out[OUTPUT_COLUMNS]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> dict:
    """
    Read Bronze, apply the Silver rules, and overwrite spi_silver_tax_cleaned.
    Returns:
        dict: rows_written.
    """
    bronze = read_bronze()
    silver = transform(bronze)
    write_deltalake(TARGET_TABLE, silver, mode="overwrite", storage_options=storage_options())
    log.info(f"Data stored in spi_silver_tax_cleaned: {len(silver)} rows")
    return {"rows_written": len(silver)}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to the logging Script activity (ADR-009).

result = main()
notebookutils.notebook.exit(json.dumps(result))